# Tier-1 Pilot Extension

The initial pilot at 32 steps found only marginal improvement from rejection (best: −0.33% FID with max_prob + cap=0.2). We also learned that τ had no effect because the `max_reject_rate` cap was always binding.

This notebook runs 9 targeted configs to check whether the idea is fundamentally cooked or just underpowered. Three experiments:

| Experiment | Question | Configs |
|------------|----------|---------|
| **A. Wider cap** | Does looser cap let rejection help? | 3: max_prob, τ=0.5, cap ∈ {0.05, 0.3, 0.5} at 32 steps |
| **B. Refinement ablation** | Does post-hoc correction work where during-generation rejection didn't? | 2: max_prob, K ∈ {0.1, 0.2} at 32 steps |
| **C. Step-count sensitivity** | Does rejection help at different step counts? | 4: vanilla + rejection (cap=0.2) at steps ∈ {16, 64} |

**Total runtime: ~2.75 hours** on A100.

Results are merged with the original `pilot-20260421/results.csv` for unified analysis.

## 1. Setup

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("No GPU! Runtime > Change runtime type > A100 GPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ARPG = "/content/drive/MyDrive/ARPG-assets"

# Confirm the original pilot CSV is available
import os
ORIG_CSV = f"{DRIVE_ARPG}/results/pilot-20260421/results.csv"
assert os.path.exists(ORIG_CSV), f"Missing original pilot CSV at {ORIG_CSV}"
print(f"Found original pilot results: {ORIG_CSV}")

In [ ]:
os.chdir('/content')
!rm -rf /content/ARPG
!git clone https://github.com/rshahbazov23/comp447-arpg-private.git /content/ARPG
%cd /content/ARPG

# Symlink heavy assets
!ln -sfn {DRIVE_ARPG}/weights weights
!ln -sfn {DRIVE_ARPG}/eval eval
!ln -sfn {DRIVE_ARPG}/external external

!pip install -q transformers einops Pillow tqdm numpy scipy tensorflow pandas

In [ ]:
# Sanity check
required = [
    "weights/arpg_300m.pt",
    "weights/vq_ds16_c2i.pt",
    "eval/VIRTUAL_imagenet256_labeled.npz",
    "external/guided-diffusion/evaluations/evaluator.py",
    "scripts/run_tier1_extension.sh",
    "scripts/eval_pilot_sweep.py",
]
for p in required:
    status = "OK" if os.path.exists(p) else "MISSING"
    print(f"  [{status}] {p}")

## 2. Run the tier-1 sweep (sampling)

**~2 hours** on A100. Generates 9 new NPZs.

Safe to interrupt — re-running skips configs that already have an NPZ.

In [ ]:
TIER1_DIR = "/content/tier1-extension"
!mkdir -p {TIER1_DIR}

env = {
    'TIER1_DIR':           TIER1_DIR,
    'NUM_SAMPLES':         '10000',
    'PER_PROC_BATCH_SIZE': '32',
    'CFG_SCALE':           '5.0',
    'GLOBAL_SEED':         '0',
}
env_prefix = ' '.join(f'{k}={v}' for k, v in env.items())

!{env_prefix} bash scripts/run_tier1_extension.sh

In [ ]:
# Verify we have 9 NPZs (3 vanilla-not, 4 vanilla+rejection for step C, ... actually let me recount)
# A: 3 rejection configs (32 steps, wider cap)
# B: 2 refinement configs (32 steps)
# C: 4 configs — 2 vanilla (16, 64 steps) + 2 rejection (16, 64 steps)
# Total: 9
import glob
vanilla_npzs = sorted(glob.glob(f"{TIER1_DIR}/vanilla/*.npz"))
rejection_npzs = sorted(glob.glob(f"{TIER1_DIR}/rejection/*.npz"))
refinement_npzs = sorted(glob.glob(f"{TIER1_DIR}/refinement/*.npz"))

print(f"Vanilla NPZs (steps 16, 64):   {len(vanilla_npzs)} / 2")
print(f"Rejection NPZs (A + C):         {len(rejection_npzs)} / 5")
print(f"Refinement NPZs (B):            {len(refinement_npzs)} / 2")
!du -sh {TIER1_DIR}

## 3. FID evaluation on the 9 new NPZs

**~45 minutes** — 9 configs × ~5 min each.

In [ ]:
TIER1_CSV = f"{TIER1_DIR}/tier1-results.csv"

!python scripts/eval_pilot_sweep.py \
    --pilot-dir {TIER1_DIR} \
    --reference-npz eval/VIRTUAL_imagenet256_labeled.npz \
    --guided-diffusion external/guided-diffusion/evaluations/evaluator.py \
    --out-csv {TIER1_CSV}

## 4. Merge results with the original pilot

Load both CSVs, tag each row with its experimental source, and analyse.

In [ ]:
import pandas as pd
import re

def extract_steps(npz_name):
    m = re.search(r'step-(\d+)', npz_name)
    return int(m.group(1)) if m else None

orig = pd.read_csv(ORIG_CSV)
orig['steps'] = orig['npz'].apply(extract_steps)
orig['source'] = 'pilot'

new = pd.read_csv(TIER1_CSV)
new['steps'] = new['npz'].apply(extract_steps)
new['source'] = 'tier1'

all_results = pd.concat([orig, new], ignore_index=True)
all_results = all_results[all_results['fid'].notna()].copy()

print(f"Original pilot rows: {len(orig)}")
print(f"Tier-1 rows: {len(new)}")
print(f"Combined successful rows: {len(all_results)}")

### 4.1 Experiment A — Wider cap grid

Compare max_prob + τ=0.5 across cap ∈ {0.05, 0.1, 0.2, 0.3, 0.5} at 32 steps. The existing pilot contributes cap ∈ {0.1, 0.2}; tier-1 contributes {0.05, 0.3, 0.5}.

In [ ]:
exp_a = all_results[
    (all_results['mode'] == 'rejection') &
    (all_results['metric'] == 'max_prob') &
    (all_results['tau'] == 0.5) &
    (all_results['steps'] == 32)
].copy().sort_values('cap')

vanilla_32 = all_results[(all_results['mode'] == 'vanilla') & (all_results['steps'] == 32)]['fid'].iloc[0]
print(f"Vanilla FID at 32 steps: {vanilla_32:.3f}\n")

print("=== Experiment A: cap sweep (max_prob, tau=0.5, 32 steps) ===")
print(f"{'cap':>6} {'FID':>8} {'\u0394 vs vanilla':>14}")
print("-" * 36)
for _, r in exp_a.iterrows():
    delta = r['fid'] - vanilla_32
    marker = "  <--" if delta < 0 else ""
    print(f"{r['cap']:>6.2f} {r['fid']:>8.3f} {delta:>+14.3f}{marker}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(exp_a['cap'], exp_a['fid'], marker='o', linewidth=2, markersize=8, label='max_prob, tau=0.5')
ax.axhline(vanilla_32, color='gray', linestyle='--', label=f'vanilla (32 steps) = {vanilla_32:.3f}')
ax.set_xlabel('max_reject_rate (cap)')
ax.set_ylabel('FID-10K (lower is better)')
ax.set_title('Experiment A — cap sweep at 32 steps, max_prob, tau=0.5')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 4.2 Experiment B — Refinement ablation

Does post-hoc re-decoding of low-confidence tokens (full KV cache) beat during-generation rejection?

In [ ]:
exp_b = all_results[
    (all_results['mode'] == 'refinement') &
    (all_results['metric'] == 'max_prob') &
    (all_results['steps'] == 32)
].copy().sort_values('cap')  # `cap` column holds refinement_k for refinement rows

best_rejection_32 = all_results[
    (all_results['mode'] == 'rejection') &
    (all_results['steps'] == 32)
]['fid'].min()

print(f"Vanilla FID (32 steps):           {vanilla_32:.3f}")
print(f"Best rejection (32 steps):        {best_rejection_32:.3f}")
print()
print("=== Experiment B: refinement ablation (max_prob, 32 steps) ===")
print(f"{'K':>6} {'FID':>8} {'\u0394 vs vanilla':>14} {'vs best rejection':>20}")
print("-" * 56)
for _, r in exp_b.iterrows():
    k = r['cap']  # stored as 'cap' column
    delta_v = r['fid'] - vanilla_32
    delta_r = r['fid'] - best_rejection_32
    v_marker = "*" if delta_v < 0 else " "
    r_marker = "*" if delta_r < 0 else " "
    print(f"{k:>6.2f} {r['fid']:>8.3f} {delta_v:>+12.3f}{v_marker} {delta_r:>+18.3f}{r_marker}")

### 4.3 Experiment C — Step-count sensitivity

Does rejection help more (or less) at different step counts?

In [ ]:
exp_c_vanilla = all_results[all_results['mode'] == 'vanilla'].copy().sort_values('steps')
exp_c_reject  = all_results[
    (all_results['mode'] == 'rejection') &
    (all_results['metric'] == 'max_prob') &
    (all_results['tau'] == 0.5) &
    (all_results['cap'] == 0.2)
].copy().sort_values('steps')

print("=== Experiment C: step-count sensitivity (max_prob, tau=0.5, cap=0.2) ===")
print(f"{'steps':>6} {'vanilla FID':>12} {'rejection FID':>15} {'delta':>10}")
print("-" * 48)
for steps in [16, 32, 64]:
    v_rows = exp_c_vanilla[exp_c_vanilla['steps'] == steps]
    r_rows = exp_c_reject[exp_c_reject['steps'] == steps]
    v = v_rows['fid'].iloc[0] if len(v_rows) else None
    r = r_rows['fid'].iloc[0] if len(r_rows) else None
    if v is not None and r is not None:
        delta = r - v
        marker = "  <--" if delta < 0 else ""
        print(f"{steps:>6} {v:>12.3f} {r:>15.3f} {delta:>+10.3f}{marker}")
    else:
        print(f"{steps:>6} {str(v):>12} {str(r):>15}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

v_data = exp_c_vanilla.sort_values('steps')
r_data = exp_c_reject.sort_values('steps')

ax.plot(v_data['steps'], v_data['fid'], marker='s', linewidth=2, markersize=10, label='vanilla', color='tab:gray')
ax.plot(r_data['steps'], r_data['fid'], marker='o', linewidth=2, markersize=10, label='rejection (max_prob, cap=0.2)', color='tab:blue')
ax.set_xlabel('decoding steps')
ax.set_ylabel('FID-10K (lower is better)')
ax.set_title('Experiment C — FID vs decoding steps')
ax.set_xticks([16, 32, 64])
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Verdict

The following cell prints a short diagnostic summary across all three experiments.

In [ ]:
print("=" * 70)
print("TIER-1 DIAGNOSTIC SUMMARY")
print("=" * 70)

# Best from each experiment
best_a_row = exp_a.loc[exp_a['fid'].idxmin()]
best_a_delta = best_a_row['fid'] - vanilla_32
print(f"\nExperiment A (wider cap):")
print(f"  Best: cap={best_a_row['cap']:.2f}, FID={best_a_row['fid']:.3f}, delta vs vanilla = {best_a_delta:+.3f}")

if len(exp_b):
    best_b_row = exp_b.loc[exp_b['fid'].idxmin()]
    best_b_delta = best_b_row['fid'] - vanilla_32
    print(f"\nExperiment B (refinement):")
    print(f"  Best: K={best_b_row['cap']:.2f}, FID={best_b_row['fid']:.3f}, delta vs vanilla = {best_b_delta:+.3f}")

print(f"\nExperiment C (step-count sensitivity):")
for steps in [16, 32, 64]:
    v_rows = exp_c_vanilla[exp_c_vanilla['steps'] == steps]
    r_rows = exp_c_reject[exp_c_reject['steps'] == steps]
    if len(v_rows) and len(r_rows):
        v = v_rows['fid'].iloc[0]
        r = r_rows['fid'].iloc[0]
        print(f"  steps={steps}: vanilla={v:.3f}  rejection={r:.3f}  delta={r-v:+.3f}")

print()
print("=" * 70)
print("VERDICT HEURISTIC")
print("=" * 70)
best_deltas = []
best_deltas.append(('A: cap sweep', best_a_delta))
if len(exp_b):
    best_deltas.append(('B: refinement', best_b_delta))
for steps in [16, 64]:
    v_rows = exp_c_vanilla[exp_c_vanilla['steps'] == steps]
    r_rows = exp_c_reject[exp_c_reject['steps'] == steps]
    if len(v_rows) and len(r_rows):
        v = v_rows['fid'].iloc[0]
        r = r_rows['fid'].iloc[0]
        best_deltas.append((f'C: steps={steps}', r - v))

best_overall = min(best_deltas, key=lambda x: x[1])
print(f"\nBiggest win: {best_overall[0]}  (delta = {best_overall[1]:+.3f})")

if best_overall[1] < -0.1:
    print("\n🟢 STRONG SIGNAL. There's a real improvement worth pursuing.")
    print("   Recommend: full FID-50K on the winning config for Phase 2.")
elif best_overall[1] < -0.05:
    print("\n🟡 MODERATE SIGNAL. Improvement is small but consistent.")
    print("   Recommend: run with multiple seeds to verify it's not noise.")
elif best_overall[1] < 0:
    print("\n🟠 WEAK SIGNAL. Improvement is within noise range.")
    print("   Recommend: proceed to Tier 2 (per-sample decisions) or pivot to calibration story.")
else:
    print("\n🔴 NO SIGNAL. No config beats vanilla.")
    print("   Recommend: pivot to Tier 2 (per-sample) or reframe as calibration/negative-result paper.")

## 6. Save to Drive

In [ ]:
DRIVE_RESULTS = f"{DRIVE_ARPG}/results/pilot-20260421/tier1-extension"
!mkdir -p {DRIVE_RESULTS}/logs

# Save the new CSV + logs + heatmaps (not the NPZs)
!cp {TIER1_CSV} {DRIVE_RESULTS}/tier1-results.csv
!cp {TIER1_DIR}/logs/*.json {DRIVE_RESULTS}/logs/ 2>/dev/null || echo "(no JSON logs)"
!cp {TIER1_DIR}/logs/*_heatmap.png {DRIVE_RESULTS}/logs/ 2>/dev/null || echo "(no heatmaps)"

# Also save the merged analysis CSV
all_results.to_csv(f"{TIER1_DIR}/combined-results.csv", index=False)
!cp {TIER1_DIR}/combined-results.csv {DRIVE_RESULTS}/combined-results.csv

print(f"\nResults backed up to: {DRIVE_RESULTS}")
!ls -lh {DRIVE_RESULTS}